# Fine-Tuning de SLMs para Detección de Botnets (QLoRA)
Este notebook aplica la técnica de parametrización eficiente QLoRA (Quantized Low-Rank Adaptation) a un conjunto de iteraciones de modelos SLM orientados a edge computing usando cuantización a 4-bit.

## 1. Instalación, Importación y Autenticación

In [ ]:
!pip install -q kagglehub transformers datasets peft accelerate scikit-learn psutil pandas numpy huggingface_hub
!pip install -q -U bitsandbytes>=0.46.1

import kagglehub
import pandas as pd
import numpy as np
import os, time, psutil, gc, glob
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, matthews_corrcoef, precision_score, f1_score
from sklearn.preprocessing import RobustScaler
from transformers import AutoModel, AutoConfig, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Autenticación para Modelos Gated (ej. Gemma)
from google.colab import userdata
from huggingface_hub import login
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Autenticación exitosa en HuggingFace.")
except Exception as e:
    print("⚠️ Advertencia: No se encontró HF_TOKEN. Modelos protegidos podrían fallar.")

## 2. Variables Globales e Hiperparámetros

In [ ]:
# ==========================================
# HIPERPARÁMETROS DEL BENCHMARK
# ==========================================
MUESTRAS_BENIGN = 5000
MUESTRAS_ATAQUE = 2000
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_TEST = 128
TEST_SIZE_SPLIT = 0.2

NUM_CLASSES = 3
EPOCHS = 1
LEARNING_RATE = 1e-4

LORA_R = 8
LORA_ALPHA = 32
LORA_TARGET_MODULES = "all-linear"

RANDOM_SEED = 42

FEATURES_UNIVERSAL = [
    'HH_jit_L0.1_mean', 'MI_dir_L5_weight', 'H_L5_weight', 'HpHp_L0.01_radius',
    'HpHp_L0.01_std', 'HH_L0.1_radius', 'HH_L0.01_std', 'HH_L0.1_weight',
    'MI_dir_L0.1_mean', 'HH_jit_L5_mean', 'HpHp_L0.1_weight', 'HH_jit_L0.1_weight',
    'HH_L0.01_covariance', 'HpHp_L0.1_radius', 'HpHp_L5_magnitude'
]

slm_benchmark_list = [
    {"name": "DistilRoBERTa", "id": "distilroberta-base"},
    {"name": "Qwen2.5-0.5B", "id": "Qwen/Qwen2.5-0.5B-Instruct"},
    {"name": "Phi-1.5", "id": "microsoft/phi-1_5"},
    {"name": "TinyLlama-1.1B", "id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"},
    {"name": "SmolLM-360M", "id": "HuggingFaceTB/SmolLM-360M-Instruct"},
    {"name": "Gemma-4-E2B", "id": "google/gemma-4-E2B-it"}
]

## 3. Arquitectura SLM Adaptada a Series Temporales (QLoRA 4-bit)

In [ ]:
class SLMEdgeQLoRA(nn.Module):
    def __init__(self, checkpoint, num_features, num_classes=NUM_CLASSES):
        super().__init__()
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        
        self.config = AutoConfig.from_pretrained(checkpoint, trust_remote_code=True)
        if getattr(self.config, "pad_token_id", None) is None:
            self.config.pad_token_id = getattr(self.config, "eos_token_id", 0)

        self.transformer = AutoModel.from_pretrained(
            checkpoint, 
            config=self.config, 
            trust_remote_code=True,
            quantization_config=bnb_config, 
            device_map="auto"
        )

        self.hidden_size = getattr(self.config, "hidden_size", getattr(self.config, "d_model", 768))
        self.embed_size = getattr(self.config, "embedding_size", self.hidden_size)

        self.feature_projector = nn.Linear(1, self.embed_size)
        self.feature_embeddings = nn.Parameter(torch.randn(1, num_features, self.embed_size))
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.hidden_size),
            nn.Linear(self.hidden_size, num_classes)
        )

    def forward(self, x):
        device = next(self.transformer.parameters()).device
        dtype = next(self.transformer.parameters()).dtype

        x = x.unsqueeze(-1).to(device).to(torch.float32)
        tokens = (self.feature_projector.to(device)(x) + self.feature_embeddings.to(device)).to(dtype)

        if getattr(self.config, "is_encoder_decoder", False):
            outputs = self.transformer.encoder(inputs_embeds=tokens)
        else:
            outputs = self.transformer(inputs_embeds=tokens)
            
        hidden_states = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs[0]
        pooled = hidden_states.mean(dim=1).to(torch.float32)
        return self.classifier.to(device)(pooled)

## 4. Pipeline de Datos y Métricas

In [ ]:
def preparar_datos_rapidos(path):
    def cargar_eficiente(patron, label, n):
        archivos = glob.glob(os.path.join(path, '**', patron), recursive=True)
        if not archivos: return pd.DataFrame()
        try:
            df = pd.read_csv(archivos[0], usecols=FEATURES_UNIVERSAL, nrows=n).dropna()
            df['label'] = label
            return df
        except:
            return pd.DataFrame()

    print("⚡ Cargando datos del dataset N-BaIoT...")
    df_total = pd.concat([
        cargar_eficiente('*.benign.csv', 0, MUESTRAS_BENIGN),
        cargar_eficiente('*.mirai.*.csv', 1, MUESTRAS_ATAQUE),
        cargar_eficiente('*.gafgyt.*.csv', 2, MUESTRAS_ATAQUE)
    ], ignore_index=True)

    labels = df_total.pop('label').values
    scaler = RobustScaler()
    X = scaler.fit_transform(df_total.values)

    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=TEST_SIZE_SPLIT, stratify=labels, random_state=RANDOM_SEED)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).long()), batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).long()), batch_size=BATCH_SIZE_TEST, shuffle=False)
    return train_loader, test_loader

def capturar_metricas(model, loader, device, nombre_modelo, fase, epoch):
    model.eval()
    y_true, y_pred = [], []
    mem_uso = psutil.Process().memory_info().rss / (1024 * 1024)
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters()) / (1024 * 1024)

    inicio_t = time.perf_counter()
    with torch.no_grad():
        for bx, by in loader:
            outputs = model(bx.to(device))
            y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            y_true.extend(by.numpy())

    tiempo_ms = ((time.perf_counter() - inicio_t) / len(loader.dataset)) * 1000
    return {
        "Modelo": nombre_modelo, "Fase": fase, "Epoch": epoch,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='macro', zero_division=0),
        "F1-Macro": f1_score(y_true, y_pred, average='macro', zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "RAM_MB": mem_uso, "Storage_MB": param_size, "Infer_ms": tiempo_ms
    }

## 5. Bucle de Entrenamiento Principal

In [ ]:
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
train_loader, test_loader = preparar_datos_rapidos(path)

device = "cuda" if torch.cuda.is_available() else "cpu"
historico_final = []

for slm in slm_benchmark_list:
    try:
        nombre = slm['name']
        estado_txt = "QLoRA_4bit"
        print(f"\n--- Iniciando Evaluación: {nombre} | MODO: {estado_txt} ---")
        model = SLMEdgeQLoRA(slm['id'], len(FEATURES_UNIVERSAL))

        # Setup QLoRA
        model.transformer = prepare_model_for_kbit_training(model.transformer)
        peft_config = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES, task_type=TaskType.FEATURE_EXTRACTION
        )
        model.transformer = get_peft_model(model.transformer, peft_config)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(1, EPOCHS + 1):
            # Entrenamiento con Barra de Progreso
            model.train()
            pbar = tqdm(train_loader, desc=f"Training {nombre} (Ep {epoch})", unit="batch")

            for bx, by in pbar:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                optimizer.step()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            # Capture metrics explicitly at epoch end
            res = capturar_metricas(model, test_loader, device, nombre, estado_txt, epoch)
            historico_final.append(res)
            print(f"✅ QLoRA - Epoch {epoch} | F1: {res['F1-Macro']:.4f} | RAM: {res['RAM_MB']:.2f} MB")

        # Guardado local del adaptador QLoRA antes de limpiarlo de RAM
        os.makedirs(f"../modelos_entrenados/{nombre}_qlora", exist_ok=True)
        model.save_pretrained(f"../modelos_entrenados/{nombre}_qlora")
        print(f"💾 Guardado localmente: ../modelos_entrenados/{nombre}_qlora")

        # Limpieza Crítica de Memoria
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()
        time.sleep(2) # Respiro para el hardware

    except Exception as e:
        print(f"❌ Error crítico en {slm['name']}: {e}")
        gc.collect()
        torch.cuda.empty_cache()

df_final = pd.DataFrame(historico_final)
df_final.to_csv("benchmark_iot_qlora.txt", sep="\t", index=False)
print("\n✨ Benchmark finalizado con éxito. Resultados guardados en 'benchmark_iot_qlora.txt'")

## 6. Visualización de Resultados

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Preparar los datos leyendo del archivo TXT generado
df = pd.read_csv('benchmark_iot_qlora.txt', sep='\t')

# Extraer solo los resultados del último Epoch
df = df[df['Epoch'] == df['Epoch'].max()].copy()

# Renombrar las columnas para visualización
df = df.rename(columns={
    'Fase': 'Tipo',
    'RAM_MB': 'RAM (MB)',
    'Storage_MB': 'Storage (MB)',
    'Infer_ms': 'Inferencia (ms)'
})

# Agrupar para los labels de los gráficos
df['Modelo_Modo'] = df['Modelo'] + ' (' + df['Tipo'] + ')'
df = df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(4, 1, figsize=(14, 28))

# --- GRÁFICO 1: MÉTRICAS DE CALIDAD ---
df_melted = df.melt(id_vars='Modelo_Modo', value_vars=['Accuracy', 'F1-Macro', 'MCC'],
                    var_name='Métrica', value_name='Valor')

sns.barplot(data=df_melted, x='Modelo_Modo', y='Valor', hue='Métrica', ax=axes[0], palette='viridis')
axes[0].set_title('Comparativa de Calidad de Modelos QLoRA: Accuracy, F1-Macro y MCC', fontsize=16, fontweight='bold')
axes[0].set_ylim(0.89, 1.0) 
axes[0].set_ylabel('Puntuación')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(loc='upper right')

# --- GRÁFICO 2: EFICIENCIA VS ALMACENAMIENTO ---
sns.scatterplot(data=df, x='Inferencia (ms)', y='Storage (MB)', hue='Modelo_Modo',
                size='Accuracy', sizes=(100, 500), ax=axes[1], palette='tab20')
for i in range(df.shape[0]):
    axes[1].text(df['Inferencia (ms)'][i] + 0.1, df['Storage (MB)'][i],
                 df['Modelo_Modo'][i], fontsize=8, verticalalignment='center')
axes[1].set_title('Eficiencia: Tiempo de Inferencia vs Peso en Disco (QLoRA)', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Inferencia (ms)')
axes[1].set_ylabel('Almacenamiento (MB)')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., title='Modelos')

# --- GRÁFICO 3: CONSUMO DE RAM ---
df_ram_sorted = df.sort_values('RAM (MB)')
sns.barplot(data=df_ram_sorted, x='Modelo_Modo', y='RAM (MB)', ax=axes[2], palette='magma')
axes[2].set_title('Consumo de Memoria RAM máximo', fontsize=16, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylabel('RAM (MB)')

# --- GRÁFICO 4: ZOOM DE EFICIENCIA ---
sns.scatterplot(data=df, x='Inferencia (ms)', y='Storage (MB)', hue='Modelo_Modo',
                size='Accuracy', sizes=(200, 800), ax=axes[3], palette='rocket_r', legend=False)

for i in range(df.shape[0]):
    if (0 <= df['Inferencia (ms)'][i] <= 3) and (0 <= df['Storage (MB)'][i] <= 1000):
        modo_corto = f"({df['Tipo'][i][0]})"
        label_corta = f"{df['Modelo'][i]} {modo_corto}"
        axes[3].text(df['Inferencia (ms)'][i] + 0.05, df['Storage (MB)'][i] + 30,
                     label_corta, fontsize=9, verticalalignment='center')

axes[3].set_xlim(-0.5, 2.5)
axes[3].set_ylim(-100, 800)
axes[3].set_title('Eficiencia (Zoom): Inferencia vs Almacenamiento', fontsize=16, fontweight='bold')
axes[3].set_xlabel('Inferencia (ms)')
axes[3].set_ylabel('Almacenamiento (MB)')

plt.tight_layout()
plt.savefig('comparativa_qlora_4_graficos.png')
plt.show()